# Modeling and Associations

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

cwd = Path.cwd()
root = next(p for p in [cwd, *cwd.parents] if (p / "data" / "raw" / "survey_data.csv").exists())
df_clean = pd.read_csv(root / "data" / "clean" / "clean_data.csv")

## Prepare Ordinal Variables

In [2]:
num = df_clean.copy()

# Codes 1-5 remain in codebook direction. Code 5 is excluded as Don't know/Unsure.
ordinal = ["q1", "q2", "q3", "q5"]
for q in ordinal:
    num[q] = pd.to_numeric(num[q], errors="raise").astype(float)
    num[q] = num[q].where(num[q].between(1, 4))

for c in ["location", "sex", "q4"]:
    num[c] = num[c].astype(str)

print("Missing values after excluding code 5:")
print(num[ordinal].isna().sum())

Missing values after excluding code 5:
q1    12
q2    10
q3     1
q5    24
dtype: int64


## Categorical Associations

In [3]:
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

rows = []
for group in ["location", "sex"]:
    for question in ["q1", "q2", "q3", "q4", "q5"]:
        sub = num[[group, question]].dropna()
        ct = pd.crosstab(sub[group], sub[question])
        row = {"group": group, "question": question, "n": len(sub), "chi2_p": stats.chi2_contingency(ct)[1], "cramers_v": cramers_v(ct)}
        if question != "q4":
            groups = [values.to_numpy(dtype=float) for _, values in sub.groupby(group)[question]]
            if len(groups) == 2:
                u = stats.mannwhitneyu(groups[0], groups[1])
                row["mw_p"] = u.pvalue
                row["prob_superiority"] = u.statistic / (len(groups[0]) * len(groups[1]))
        rows.append(row)

results = pd.DataFrame(rows)
results["p_holm"] = multipletests(results["chi2_p"], method="holm")[1]
display(results.round(3))

,group,question,n,chi2_p,cramers_v,mw_p,prob_superiority,p_holm
0,location,q1,88,0.000,0.508,0.000,0.229,0.000
1,location,q2,90,0.081,0.273,0.031,0.366,0.406
2,location,q3,99,0.317,0.189,0.329,0.445,0.634
3,location,q4,100,0.002,0.410,NaN,NaN,0.017
4,location,q5,76,0.151,0.264,0.169,0.402,0.604
5,sex,q1,88,0.155,0.244,0.124,0.403,0.604
6,sex,q2,90,0.031,0.315,0.008,0.336,0.214
7,sex,q3,99,0.563,0.144,0.173,0.424,0.634
8,sex,q4,100,0.001,0.437,NaN,NaN,0.007
9,sex,q5,76,0.060,0.312,0.048,0.367,0.359


## Logistic Models of Top-2 Responses

In [4]:
for q in ["q1", "q2", "q3", "q5"]:
    d = num[[q, "location", "sex"]].dropna().copy()
    d["top2"] = d[q].le(2).astype(int)
    model = smf.logit("top2 ~ C(location) + C(sex)", d).fit(disp=0)
    table = pd.DataFrame({"OR": np.exp(model.params), "p": model.pvalues})
    print(q, "n =", len(d))
    display(table.round(3))

q1 n = 88


,OR,p
Intercept,1.833,0.047
C(location)[T.MT],0.147,0.000
C(sex)[T.M],0.861,0.770


q2 n = 90


,OR,p
Intercept,2.174,0.012
C(location)[T.MT],0.721,0.493
C(sex)[T.M],0.356,0.029


q3 n = 99


,OR,p
Intercept,3.247,0.000
C(location)[T.MT],1.290,0.622
C(sex)[T.M],0.557,0.240


q5 n = 76


,OR,p
Intercept,1.427,0.245
C(location)[T.MT],0.666,0.485
C(sex)[T.M],0.300,0.030


## Spearman Associations with Donation Likelihood

In [5]:
rows = []
for q in ["q1", "q2", "q3"]:
    d = num[[q, "q5"]].dropna()
    rho, p = stats.spearmanr(d[q].to_numpy(float), d["q5"].to_numpy(float))
    rows.append({"awareness": q, "n": len(d), "rho": rho, "p": p})
display(pd.DataFrame(rows).round(3))

,awareness,n,rho,p
0,q1,65,0.233,0.061
1,q2,70,0.336,0.004
2,q3,75,0.443,0.000
